<a href="https://colab.research.google.com/github/Priya-Kumari-Chourasia/Pytorch_tutorials/blob/main/pytorch_training_pipeline_using_nn_module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
import torch.nn as nn

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/breast-cancer-dataset")

print("Path to dataset files:", path)

100%|██████████| 48.6k/48.6k [00:00<00:00, 2.53MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/yasserh/breast-cancer-dataset/versions/1


In [4]:
df = pd.read_csv(f"{path}/breast-cancer.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [5]:
df.drop(columns=['id'],inplace=True)

In [6]:
x_train,x_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [7]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [8]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [9]:
x_train_tensor = torch.from_numpy(x_train)
x_test_tensor = torch.from_numpy(x_test)

y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [10]:
x_train_tensor.dtype

torch.float64

In [11]:
class MySimple(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear = nn.Linear(num_features,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,features):
    out = self.linear(features)
    out = self.sigmoid(out)

    return out

  def loss(self,y_pred,y):
    epsilon= 1e-7
    y_pred = torch.clamp(y_pred,epsilon,1-epsilon)

    loss = -(y_train_tensor*torch.log(y_pred) + (1-y_train_tensor)*(1-y_pred)).mean()
    return loss

In [12]:
learning_rate = 0.1
epochs=25

In [13]:
x_train_tensor = x_train_tensor.float()
y_train_tensor = y_train_tensor.float()
model = MySimple(x_train_tensor.shape[1])

for epoch in range(epochs):
  y_pred= model(x_train_tensor)
  loss = model.loss(y_pred,y_train_tensor)

  loss.backward()

  with torch.no_grad():
    model.linear.weight -= learning_rate +model.linear.weight.grad
    model.linear.bias -= learning_rate + model.linear.bias.grad

  model.linear.weight.grad.zero_()
  model.linear.bias.grad.zero_()

  print(f'Epoch:{epoch+1},Loss : {loss.item()}')



Epoch:1,Loss : -0.04457147791981697
Epoch:2,Loss : 0.1048700362443924
Epoch:3,Loss : 0.13205978274345398
Epoch:4,Loss : 0.1598842740058899
Epoch:5,Loss : 0.1885252445936203
Epoch:6,Loss : 0.21734870970249176
Epoch:7,Loss : 0.24601705372333527
Epoch:8,Loss : 0.2744080126285553
Epoch:9,Loss : 0.3024432063102722
Epoch:10,Loss : 0.3346574902534485
Epoch:11,Loss : 0.366006076335907
Epoch:12,Loss : 0.3967285454273224
Epoch:13,Loss : 0.4266955256462097
Epoch:14,Loss : 0.46105414628982544
Epoch:15,Loss : 0.49753832817077637
Epoch:16,Loss : 0.5352892875671387
Epoch:17,Loss : 0.5739836096763611
Epoch:18,Loss : 0.621221661567688
Epoch:19,Loss : 0.6668341755867004
Epoch:20,Loss : 0.7150372862815857
Epoch:21,Loss : 0.7616328597068787
Epoch:22,Loss : 0.8104305863380432
Epoch:23,Loss : 0.8601627349853516
Epoch:24,Loss : 0.9123096466064453
Epoch:25,Loss : 0.9691579937934875


In [14]:
# using builtin loss function

loss_function  = nn.BCELoss()

In [16]:
x_train_tensor = x_train_tensor.float()
y_train_tensor = y_train_tensor.float()
model = MySimple(x_train_tensor.shape[1])

for epoch in range(epochs):
  y_pred= model(x_train_tensor)
  loss = loss_function(y_pred,y_train_tensor.view(-1,1))

  loss.backward()

  with torch.no_grad():
    model.linear.weight -= learning_rate +model.linear.weight.grad
    model.linear.bias -= learning_rate + model.linear.bias.grad

  model.linear.weight.grad.zero_()
  model.linear.bias.grad.zero_()

  print(f'Epoch:{epoch+1},Loss : {loss.item()}')



Epoch:1,Loss : 0.8427837491035461
Epoch:2,Loss : 0.15433651208877563
Epoch:3,Loss : 0.23250028491020203
Epoch:4,Loss : 0.29977551102638245
Epoch:5,Loss : 0.29663947224617004
Epoch:6,Loss : 0.3028036057949066
Epoch:7,Loss : 0.30459779500961304
Epoch:8,Loss : 0.30742424726486206
Epoch:9,Loss : 0.3098285496234894
Epoch:10,Loss : 0.31241172552108765
Epoch:11,Loss : 0.3150383532047272
Epoch:12,Loss : 0.31774088740348816
Epoch:13,Loss : 0.3204977214336395
Epoch:14,Loss : 0.3232991397380829
Epoch:15,Loss : 0.32613441348075867
Epoch:16,Loss : 0.3289954662322998
Epoch:17,Loss : 0.33187541365623474
Epoch:18,Loss : 0.334769070148468
Epoch:19,Loss : 0.33767229318618774
Epoch:20,Loss : 0.3405817449092865
Epoch:21,Loss : 0.34349507093429565
Epoch:22,Loss : 0.34641033411026
Epoch:23,Loss : 0.3493260443210602
Epoch:24,Loss : 0.3522414565086365
Epoch:25,Loss : 0.3551557660102844


#### Evaluation

In [18]:
with torch.no_grad():
  x_test_tensor = x_test_tensor.float()
  y_pred=model.forward(x_test_tensor)
  y_pred = (y_pred>0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5664820075035095


torch.optim used to update the parameters of model during training

In [19]:
x_train_tensor = x_train_tensor.float()
y_train_tensor = y_train_tensor.float()
model = MySimple(x_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)


for epoch in range(epochs):
  y_pred= model(x_train_tensor)
  loss = loss_function(y_pred,y_train_tensor.view(-1,1))
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print(f'Epoch:{epoch+1},Loss : {loss.item()}')



Epoch:1,Loss : 0.8557366132736206
Epoch:2,Loss : 0.6046048402786255
Epoch:3,Loss : 0.47865691781044006
Epoch:4,Loss : 0.40655991435050964
Epoch:5,Loss : 0.3595193326473236
Epoch:6,Loss : 0.32606348395347595
Epoch:7,Loss : 0.3008321225643158
Epoch:8,Loss : 0.2809831202030182
Epoch:9,Loss : 0.26486489176750183
Epoch:10,Loss : 0.2514496445655823
Epoch:11,Loss : 0.2400626838207245
Epoch:12,Loss : 0.2302417904138565
Epoch:13,Loss : 0.22165904939174652
Epoch:14,Loss : 0.21407482028007507
Epoch:15,Loss : 0.2073097974061966
Epoch:16,Loss : 0.20122669637203217
Epoch:17,Loss : 0.19571846723556519
Epoch:18,Loss : 0.19070032238960266
Epoch:19,Loss : 0.1861042082309723
Epoch:20,Loss : 0.1818745732307434
Epoch:21,Loss : 0.1779656708240509
Epoch:22,Loss : 0.17433950304985046
Epoch:23,Loss : 0.1709640622138977
Epoch:24,Loss : 0.16781224310398102
Epoch:25,Loss : 0.16486087441444397


In [20]:
with torch.no_grad():
  x_test_tensor = x_test_tensor.float()
  y_pred=model.forward(x_test_tensor)
  y_pred = (y_pred>0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5393967628479004
